# Data Manipulation with Pandas

Topics:
- GroupBy: split → apply → combine
- Merging and joining DataFrames
- Concatenating DataFrames
- Pivot tables and crosstabs
- Reshaping: melt, pivot, stack/unstack
- Rolling and expanding windows
- Handling duplicates

In [ ]:
import pandas as pd
import numpy as np

# Sample sales dataset
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'date':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'region':   np.random.choice(['North','South','East','West'], n),
    'product':  np.random.choice(['Laptop','Phone','Tablet','Watch'], n),
    'rep':      np.random.choice(['Alice','Bob','Carol','Dave'], n),
    'units':    np.random.randint(1, 50, n),
    'price':    np.random.choice([999, 499, 299, 199], n),
})
df['revenue'] = df['units'] * df['price']
df['month'] = df['date'].dt.month_name()
print(df.head())
print(df.shape)

## 1. GroupBy — Split → Apply → Combine

The most powerful operation in Pandas. Group rows sharing a value, apply a function, combine results.

In [ ]:
# Basic groupby + single aggregation
print(df.groupby('region')['revenue'].sum())
print()
print(df.groupby('region')['revenue'].mean().round(2))

In [ ]:
# Multiple aggregations with .agg()
region_stats = df.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_units=('units', 'sum'),
    num_sales=('revenue', 'count')
).round(2)
print(region_stats)

In [ ]:
# Multi-level groupby
product_region = df.groupby(['product', 'region'])['revenue'].sum().unstack()
print(product_region.round(0))

In [ ]:
# GroupBy + transform — keeps original index (great for adding group stats)
df['region_avg_revenue'] = df.groupby('region')['revenue'].transform('mean')
df['above_region_avg'] = df['revenue'] > df['region_avg_revenue']
print(df[['region','revenue','region_avg_revenue','above_region_avg']].head(10))

In [ ]:
# GroupBy + filter — keep only groups meeting a condition
big_regions = df.groupby('region').filter(lambda g: g['revenue'].sum() > 500000)
print('Regions with >500k revenue:')
print(big_regions['region'].unique())

## 2. Merging & Joining DataFrames

Like SQL joins. `merge()` is the primary tool.

| Type | SQL | What it keeps |
|------|-----|----------------|
| `inner` | INNER JOIN | Only matching rows |
| `left` | LEFT JOIN | All left rows + matching right |
| `right` | RIGHT JOIN | All right rows + matching left |
| `outer` | FULL OUTER JOIN | All rows from both |

In [ ]:
# Two tables to join
orders = pd.DataFrame({
    'order_id':   [1, 2, 3, 4, 5],
    'customer_id':[101, 102, 103, 101, 104],
    'amount':     [250, 150, 300, 175, 225]
})

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 105],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave'],
    'city':        ['NYC', 'LA', 'Chicago', 'Boston']
})

print('INNER join (only matched):')
print(pd.merge(orders, customers, on='customer_id', how='inner'))
print()
print('LEFT join (all orders):')
print(pd.merge(orders, customers, on='customer_id', how='left'))

In [ ]:
# Merge on different column names
orders2 = orders.rename(columns={'customer_id': 'cust_id'})
print(pd.merge(orders2, customers, left_on='cust_id', right_on='customer_id'))

# Merge on index
customers_idx = customers.set_index('customer_id')
print(orders.merge(customers_idx, left_on='customer_id', right_index=True))

## 3. Concatenating DataFrames

In [ ]:
q1 = pd.DataFrame({'month':['Jan','Feb','Mar'], 'sales':[100,120,110]})
q2 = pd.DataFrame({'month':['Apr','May','Jun'], 'sales':[130,125,140]})
q3 = pd.DataFrame({'month':['Jul','Aug','Sep'], 'sales':[150,145,160]})

# Vertical stack (axis=0 — default)
year = pd.concat([q1, q2, q3], ignore_index=True)
print(year)

# Horizontal stack (axis=1) — columns side by side
left  = pd.DataFrame({'name': ['Alice','Bob'], 'age': [25, 30]})
right = pd.DataFrame({'score': [88, 92],       'grade': ['B','A']})
print(pd.concat([left, right], axis=1))

## 4. Pivot Tables

Pivot tables summarise data by two dimensions — like an Excel pivot table.

In [ ]:
# Revenue by product × region
pivot = df.pivot_table(
    values='revenue',
    index='product',
    columns='region',
    aggfunc='sum',
    fill_value=0,
    margins=True,    # add 'All' row and column totals
    margins_name='Total'
)
print(pivot.round(0))

# Crosstab — frequency counts
print(pd.crosstab(df['product'], df['region']))

## 5. Reshaping: melt, pivot, stack/unstack

In [ ]:
# Wide format (one row per student, one col per subject)
wide = pd.DataFrame({
    'student': ['Alice','Bob','Carol'],
    'math':    [85, 92, 78],
    'english': [90, 88, 95],
    'science': [82, 79, 91]
})
print('Wide:')
print(wide)

# melt → long format (better for groupby and plotting)
long = wide.melt(id_vars='student', var_name='subject', value_name='score')
print('\nLong:')
print(long)

# pivot → back to wide from long
back_to_wide = long.pivot(index='student', columns='subject', values='score')
print('\nBack to wide:')
print(back_to_wide)

In [ ]:
# stack and unstack — work with MultiIndex
multi = df.groupby(['product','region'])['revenue'].sum()
print('MultiIndex Series:')
print(multi.head(8))
print('\nUnstacked (product × region):')
print(multi.unstack())
print('\nStacked back:')
print(multi.unstack().stack().head(8))

## 6. Rolling & Expanding Windows

Used for time series smoothing and cumulative statistics.

In [ ]:
# Daily sales aggregated
daily = df.groupby('date')['revenue'].sum().reset_index()
daily = daily.sort_values('date')

# 7-day rolling average
daily['rolling_7d_avg'] = daily['revenue'].rolling(window=7).mean()

# Cumulative sum
daily['cumulative_revenue'] = daily['revenue'].cumsum()

# Expanding mean (average of all data up to that point)
daily['expanding_avg'] = daily['revenue'].expanding().mean()

print(daily.head(15).to_string(index=False))

## 7. Handling Duplicates

In [ ]:
dupes = pd.DataFrame({
    'name':  ['Alice','Bob','Alice','Carol','Bob','Alice'],
    'score': [85, 92, 85, 78, 92, 90]
})

print('Duplicates:')
print(dupes.duplicated())              # True for duplicate rows
print('Count:', dupes.duplicated().sum())

print('\nDuplicate rows:')
print(dupes[dupes.duplicated(keep=False)])   # show all copies

print('\nDrop exact duplicates:')
print(dupes.drop_duplicates())

print('\nKeep first by name only:')
print(dupes.drop_duplicates(subset=['name'], keep='first'))

## Quick Summary

| Operation | Key Function |
|-----------|--------------|
| Group & aggregate | `df.groupby('col').agg(...)` |
| Add group stats | `.groupby().transform()` |
| SQL-style join | `pd.merge(left, right, on='key', how='inner')` |
| Stack vertically | `pd.concat([df1,df2], ignore_index=True)` |
| Summarise 2 dims | `df.pivot_table(values, index, columns, aggfunc)` |
| Wide → Long | `df.melt(id_vars=..., var_name=..., value_name=...)` |
| Long → Wide | `df.pivot(index, columns, values)` |
| Moving average | `.rolling(7).mean()` |
| Running total | `.cumsum()` |
| Find duplicates | `.duplicated()`, `.drop_duplicates()` |

**Next →** [04 – Reading Data](../04-data-reading/)